# Token Mixing Analysis

How tokens mix across positions through attention: mixing matrix computation, mixing speed across layers, information spread from specific positions, self-information retention, and mixing entropy.

## Why This Matters

Token mixing tracks how individual token representations evolve through the network. Understanding token-level dynamics reveals how context is integrated, when predictions form, and how different positions interact to produce the output.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.token_mixing_analysis import (
    mixing_matrix, mixing_speed, information_spread,
    self_information_retention, mixing_entropy,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

In [ ]:
result = mixing_matrix(model, tokens, layer=0)
print(f'Self-mixing mean: {result["self_mixing_mean"]:.4f}')
print(f'Off-diagonal mean: {result["off_diagonal_mean"]:.4f}')
print(f'Strongest pair: {result["strongest_mixing_pair"]}')

In [ ]:
result = mixing_speed(model, tokens)
print(f'Mixing acceleration: {result["mixing_acceleration"]:.4f}')
for p in result['per_layer']:
    print(f'  Layer {p["layer"]}: self_retention={p["self_retention"]:.4f}, mixing={p["mixing_fraction"]:.4f}')

In [ ]:
result = information_spread(model, tokens, source_pos=0, layer=0)
print(f'Reach: {result["reach"]} positions, Entropy: {result["entropy"]:.4f}')
print(f'Max receiver: pos {result["max_receiver"]}')

In [ ]:
result = self_information_retention(model, tokens)
print(f'Most retained: pos {result["most_retained"]}')
print(f'Least retained: pos {result["least_retained"]}')
print(f'Per-position retention: {np.array(result["per_position_retention"]).round(3)}')

In [ ]:
result = mixing_entropy(model, tokens)
print(f'Entropy trend: {result["entropy_trend"]:.4f}')
for p in result['per_layer']:
    print(f'  Layer {p["layer"]}: entropy={p["mean_entropy"]:.4f} '
          f'(normalized={p["normalized_entropy"]:.3f})')